### 1. Data overview

- Inspect feature types (categorical, continuous, ordinal, binary)
- Check target distribution (class balance)

## 0. Data Loading

In [2]:
import numpy as np
from helper_functions.A_loading_helpers import load_csv_data
import os
import helper_functions.C_preprocessing_helpers as ph

In [24]:
data_path = 'data'
x_train, x_test, y_train, train_ids, test_ids =  load_csv_data(data_path, sub_sample=False)

x_train_raw = x_train.copy()

In [3]:
x_train.shape


(328135, 321)

# 1. Categorizing Features - continuous or categorical

**Tutorial**:  
- You can create a dictionary using `build_feature_dictionary()` that contains tells you what features in the training data are continuous or categorical and so on.  
- Here is example usage where I retrieve the indices / positions of the features that are categorical.  
- you can tell it if you want to know:  
    - `ordinal`: so discrete scale like 1,2,3,4  
    - `categorical`: like green, red, blue  
    - `continuous`: like values between: 1 and 99  
    - `continuous_but_null_also_a_number`: like continuous but the highest number means "no answer"  
    - `not_displayed_or_unrelated`: where all values are Null or the category is things like "Call is on Landline"  
- And yes I did extract this info by hand from that god-forsaken pdf because I like suffering.


In [25]:
classes_path = 'data/feature_properties/feature_classes.json'
feature_names_path = 'data/feature_properties/feature_names.csv'

feature_classes_dictionary = ph.build_feature_dictionary(classes_path, feature_names_path)
print(feature_classes_dictionary['continuous']['indices'])

[16, 17, 18, 50, 56, 60, 63, 64, 115, 164, 198, 249, 250, 251, 252, 253, 254, 256, 277, 278, 286, 287, 292, 293, 296, 297, 298, 300, 301, 302, 303, 304, 305]



- here arrays of the feature names are produced based on data type (continuous, cat)
- as well as the indices (which columns in the numpy xrain array they correspond to)

---

- For ordinal: usually its like 1,2,3,4 and then 7 or 9 for 'dont know' or 'refused'. We replace this value with 'null'. The feature _AGEYR65 doesnt have a nice gap.
- there are variables llike _FRUITEX that tell you wether the fruit responses of that person should be excluded. 

---

## 1.1 cleaning up


### 1.1.1 Ordinal Features

for the class'ordinal' figure out where the 'gap' is and replac the highest value with Null  (1,2,3,9 --> 9 is Null because it corresponds to "no answer"

In [26]:
ordinal_indices = feature_classes_dictionary['ordinal']['indices']
 
for j in ordinal_indices:
    x_train[:, j] = ph.clean_ordinal_feature(x_train[:, j])

### 1.1.2 Continuous variables with null as a number

For the class continuous_but_null_also_a_number the highest number is sometimes "no answer" sometimes something else. Check that and replace with "null"

In [27]:
feature_classes_dictionary = ph.build_feature_dictionary(classes_path, feature_names_path)
print(feature_classes_dictionary['continuous_but_null_also_a_number']['names'])
print(feature_classes_dictionary['continuous_but_null_also_a_number']['indices'])

['HHADULT', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'ALCDAY5', 'AVEDRNK2', 'DRNK3GE5', 'MAXDRNKS', 'FRUITJU1', 'FRUIT1', 'FVBEANS', 'FVGREEN', 'FVORANG', 'VEGETAB1', 'EXEROFT1', 'EXERHMM1', 'EXEROFT2', 'EXERHMM2', 'STRENGTH', 'FLSHTMY2', 'HIVTSTD3', 'BLDSUGAR', 'FEETCHK2', 'DOCTDIAB', 'CHKHEMO3', 'LONGWTCH', 'ASTHMAGE', 'ASERVIST', 'ASDRVIST', 'ASRCHKUP', 'ASACTLIM', 'SCNTWRK1', 'ADPLEASR', 'ADDOWN', 'ADSLEEP', 'ADENERGY', 'ADEAT1', 'ADFAIL', 'ADTHINK', 'ADMOVE', 'DROCDY3_', '_DRNKWEK', 'MAXVO2_', 'FC60_', 'PAFREQ1_', 'PAFREQ2_']
[26, 28, 29, 30, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 90, 91, 93, 94, 95, 102, 106, 111, 112, 113, 114, 144, 146, 148, 149, 150, 151, 196, 207, 208, 209, 210, 211, 212, 213, 214, 263, 265, 288, 289, 294, 295]


In [28]:
# Load feature metadata
classes_path = 'data/feature_properties/feature_classes.json'
feature_names_path = 'data/feature_properties/feature_names.csv'
feature_classes_dictionary = ph.build_feature_dictionary(classes_path, feature_names_path)


# Apply cleaning
from data.feature_properties.cleaning_rules_continuous import cleaning_rules  # the big dictionary we saved earlier
x_train = ph.apply_cleaning_continuous_features(x_train, feature_classes_dictionary, cleaning_rules)

### 1.1.3 Categorical One Hot Encoding

### 1.1.4 Remove not displayyed features

In [ ]:
remove_indices = feature_classes_dictionary['not_displayed_or_unrelated']['indices']
X_cleaned = np.delete(X, remove_indices, axis=1)


NameError: name 'X' is not defined

### 2. Data quality

- Missing values (counts & % per column) --> how will we impute them? Drop, Mean or regressioN????


In [14]:
import pandas as pd

df = pd.DataFrame(x_train)
df.head(20)

,0,1,2,3,4,5,6,7,8,9,...,311,312,313,314,315,316,317,318,319,320
0,53.0,11.0,11162015.0,11.0,16.0,2015.0,1100.0,2015015629.0,2015015629.0,NaN,...,1.0,1.0,3.0,3.0,4.0,1.0,1.0,NaN,NaN,2.0
1,33.0,12.0,12152015.0,12.0,15.0,2015.0,1200.0,2015004387.0,2015004387.0,1.0,...,9.0,9.0,3.0,3.0,4.0,9.0,NaN,NaN,NaN,NaN
2,20.0,10.0,10202015.0,10.0,20.0,2015.0,1100.0,2015005638.0,2015005638.0,1.0,...,4.0,2.0,2.0,2.0,3.0,1.0,1.0,1.0,2.0,2.0
3,42.0,6.0,6182015.0,6.0,18.0,2015.0,1100.0,2015004694.0,2015004694.0,NaN,...,2.0,2.0,2.0,2.0,3.0,1.0,1.0,2.0,2.0,2.0
4,24.0,11.0,11062015.0,11.0,6.0,2015.0,1100.0,2015004673.0,2015004673.0,1.0,...,9.0,9.0,3.0,3.0,4.0,1.0,1.0,NaN,NaN,2.0
5,54.0,4.0,4082015.0,4.0,8.0,2015.0,1100.0,2015004111.0,2015004111.0,NaN,...,4.0,2.0,3.0,3.0,4.0,1.0,1.0,NaN,NaN,2.0
6,39.0,12.0,12072015.0,12.0,7.0,2015.0,1100.0,2015005521.0,2015005521.0,1.0,...,4.0,2.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,2.0
7,44.0,3.0,3072015.0,3.0,7.0,2015.0,1100.0,2015000377.0,2015000377.0,1.0,...,3.0,2.0,2.0,2.0,2.0,1.0,2.0,1.0,2.0,2.0
8,21.0,7.0,7312015.0,7.0,31.0,2015.0,1100.0,2015007052.0,2015007052.0,NaN,...,1.0,1.0,3.0,3.0,4.0,1.0,1.0,NaN,NaN,2.0
9,4.0,6.0,6232015.0,6.0,23.0,2015.0,1200.0,2015006238.0,2015006238.0,NaN,...,4.0,2.0,2.0,2.0,3.0,1.0,1.0,NaN,NaN,NaN


In [22]:
def count_missing(data):
    # Initialize the counter at 0
    missing = 0
    for row in data:
        for item in row:
            if item == '' or item is None or np.isnan(item):   # Check for missing values
                missing += 1
    return missing

# Assuming we have the following data (2D list)

total_entries = len(x_train) * len(x_train[0])   # Calculate total entries (rows * columns)
missing_count = count_missing(x_train)       # Count missing values
percentage_missing = (missing_count / total_entries) * 100  # Percentage calculation

print("Total number of missing values in the cleaned data is {} or {:.2f}%".format(missing_count, percentage_missing))

Total number of missing values in the cleaned data is 48277459 or 45.83%


In [23]:
# Assuming we have the following data (2D list)

total_entries = len(x_train_raw) * len(x_train_raw[0])   # Calculate total entries (rows * columns)
missing_count = count_missing(x_train_raw)       # Count missing values
percentage_missing = (missing_count / total_entries) * 100  # Percentage calculation

print("Total number of missing values in the raw data is {} or {:.2f}%".format(missing_count, percentage_missing))

TypeError: object of type 'builtin_function_or_method' has no len()